# Machine Learning: Global vs. Local Models

In the past 13 lessons, we have operated under a massive, unspoken assumption: we assumed you only cared about forecasting a *single* time series.

In a real enterprise environment, like Walmart or Amazon, you do not forecast "Total Global Sales." You must forecast the sales of *every single individual item, in every single store, every single day*. If you have 5,000 stores and 10,000 items per store, you have **50,000,000 distinct time series**.

How do you architect a system to forecast 50 million lines of data? Do you train 50 million separate ARIMA models? Or do you train one massive Machine Learning model to handle them all simultaneously? This is the battle between **Local Models** and **Global Models**.

Scaling time series forecasting introduces a structural dataset known in econometrics as **Panel Data** (or Longitudinal Data). You have cross-sectional entities (Stores/Items) measured over time (Days/Weeks).

Let's set up our Python environment to architect a multi-series forecasting engine.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Panel Data & Multi-Series Architecture Environment Ready.")

✅ Panel Data & Multi-Series Architecture Environment Ready.


# 1. The Local Modeling Paradigm

**Local Modeling** is the traditional classical approach (e.g., ARIMA, Holt-Winters). You slice your dataset into $N$ separate, isolated time series. You then initiate a massive `for` loop, training $N$ distinct models.

### The Mathematics of Isolation

For a specific series $i$ at time $t$, the model $f_i$ only looks at its own history:


$$y_{i,t} = f_i(y_{i,t-1}, y_{i,t-2}, \dots) + \epsilon$$

* **Pros:** Hyper-specialized. The model perfectly tunes itself to the unique quirks, trend, and volatility of that specific item.
* **Cons:** 1. **Computational Nightmare**: Training 50 million ARIMA models overnight is often physically impossible without a multi-million-dollar cloud cluster.
2. **Statistical Blindness**: If Store A runs a massive promotion and sales triple, Store B's local model cannot learn from that event. The models do not communicate.
3. **The Cold Start Problem**: If you launch a brand new product today, it has zero historical lags. A local model literally cannot be trained, and will instantly fail.

# 2. The Global Modeling Paradigm

**Global Modeling** is the modern Machine Learning approach (e.g., XGBoost, LightGBM, Deep Learning). Instead of $N$ models, you stack all 50 million time series vertically into one colossal feature matrix and train **exactly one model**.

### The Mathematics of Shared Intelligence

To prevent the model from getting confused by 50 million overlapping signals, you must introduce **Static Covariates** (Identifiers). You pass the `Store_ID`, `Item_ID`, and item characteristics (like `Category` or `Price`) as standard columns in your matrix.


$$y_{i,t} = f(y_{i,t-1}, \dots, X_{i,t}, \text{Store\_ID}_i, \text{Item\_Category}_i) + \epsilon$$

* **Pros:** 1. **Cross-Learning**: The algorithm learns universal human behaviors. If it learns that "Electronics" spike on Black Friday in New York, it automatically applies that logic to "Electronics" in Los Angeles, even if the LA store has messy data.
2. **Cold Starts**: If you launch a new "Laptop", the global model looks at the categorical feature `Category="Laptop"`, realizes it behaves like other laptops, and generates a highly accurate forecast on Day 1 with zero historical lags.
* **Cons:** It might underfit highly anomalous, bizarre outlier items because it is trying to optimize the global average loss across the entire dataset.

# 3. Architecting Panel Data in Code

Let's simulate 3 different retail stores. We will Tabularize them, and then demonstrate the architectural difference between building a Local Loop and a Global Matrix.

In [2]:
# 1. Simulate Panel Data (3 Stores over 100 days)
np.random.seed(42)
days = 100
dates = pd.date_range('2023-01-01', periods=days)

# Store A: High volume, strong weekly seasonality
store_a = 500 + np.sin(np.arange(days) * (2 * np.pi / 7)) * 100 + np.random.normal(0, 20, days)
# Store B: Medium volume, growing trend
store_b = 300 + (np.arange(days) * 2) + np.random.normal(0, 15, days)
# Store C: Low volume, erratic noise
store_c = 50 + np.random.normal(0, 10, days)

# Construct the Panel DataFrame (Stacked vertically!)
df_panel = pd.DataFrame({
    'Date': np.tile(dates, 3),
    'Store_ID': ['A']*days + ['B']*days + ['C']*days,
    'Sales': np.concatenate([store_a, store_b, store_c])
})

# 2. Tabularize the Data (Create Lags strictly grouped by Store!)
# We CANNOT just shift the whole column, or Store B's Day 1 will use Store A's Day 100 as a lag!
df_panel['Lag_1'] = df_panel.groupby('Store_ID')['Sales'].shift(1)
df_panel['Lag_7'] = df_panel.groupby('Store_ID')['Sales'].shift(7)

# Drop NaNs and sort chronologically
df_panel = df_panel.dropna().sort_values('Date')

# Convert Store_ID to numeric for ML
df_panel['Store_ID_Num'] = df_panel['Store_ID'].map({'A': 0, 'B': 1, 'C': 2})

# 3. Chronological Train/Test Split
split_date = '2023-03-20' # Leave roughly 2 weeks for testing
train = df_panel[df_panel['Date'] < split_date]
test = df_panel[df_panel['Date'] >= split_date]

# --- ARCHITECTURE A: LOCAL MODELS (The Loop) ---
local_maes = []
for store in ['A', 'B', 'C']:
    # Filter data to ONLY this specific store
    train_local = train[train['Store_ID'] == store]
    test_local = test[test['Store_ID'] == store]
    
    X_train_local = train_local[['Lag_1', 'Lag_7']]
    y_train_local = train_local['Sales']
    X_test_local = test_local[['Lag_1', 'Lag_7']]
    y_test_local = test_local['Sales']
    
    # Train an isolated model
    model_local = RandomForestRegressor(n_estimators=50, random_state=42)
    model_local.fit(X_train_local, y_train_local)
    
    preds_local = model_local.predict(X_test_local)
    local_maes.append(mean_absolute_error(y_test_local, preds_local))

print(f"🚨 Local Models Average MAE (3 Models Trained): {np.mean(local_maes):.2f}")


# --- ARCHITECTURE B: GLOBAL MODEL (The Monolith) ---
# We feed ALL stores into the matrix simultaneously, using Store_ID_Num as a feature
X_train_global = train[['Lag_1', 'Lag_7', 'Store_ID_Num']]
y_train_global = train['Sales']
X_test_global = test[['Lag_1', 'Lag_7', 'Store_ID_Num']]
y_test_global = test['Sales']

# Train exactly ONE model
model_global = RandomForestRegressor(n_estimators=50, random_state=42)
model_global.fit(X_train_global, y_train_global)

preds_global = model_global.predict(X_test_global)
global_mae = mean_absolute_error(y_test_global, preds_global)

print(f"🚨 Global Model Average MAE (1 Model Trained):  {global_mae:.2f}")
print("\nInsight: The Global model achieves highly competitive accuracy while requiring 3x less training infrastructure. At 50 million series, this efficiency becomes mandatory.")

🚨 Local Models Average MAE (3 Models Trained): 20.54
🚨 Global Model Average MAE (1 Model Trained):  13.89

Insight: The Global model achieves highly competitive accuracy while requiring 3x less training infrastructure. At 50 million series, this efficiency becomes mandatory.


# 4. The Global Architecture Advantage

Beyond pure computational speed, Global Models allow you to easily inject massive arrays of Exogenous data ($X$).

If you are a retail chain, you can add a column for `City_Population`, `Average_Income`, `Square_Footage`, and `Distance_To_Competitor`.
By passing these static features into XGBoost alongside the time-based lags, the algorithm can physically map the relationship between store demographics and sales volume. It learns that "Large stores in wealthy neighborhoods sell more Premium Coffee."

If you build a brand new store tomorrow, you simply pass its demographics into the model, and the Global intelligence will accurately forecast its grand opening sales, completely bypassing the Cold Start problem.

## Real-World Use Case or Analogy:

Think of Global vs. Local models like **Providing Healthcare**:

* **Local Models (10,000 Isolated Private Doctors)**: Every patient is assigned a doctor who sits in a locked room. The doctor learns everything about that specific patient (their heart rate, their diet). If the patient gets sick, the doctor treats them beautifully. But if Patient A gets a rare disease and the doctor cures it, the doctor treating Patient B *never finds out*. The intelligence is trapped in isolation.
* **Global Model (1 Massive Super-Hospital AI)**: Every single patient's chart is fed into one central intelligence. The hospital AI recognizes that Patient B's symptoms match exactly what happened to Patient A last month. It instantly applies the cure. It shares statistical power across the entire population, finding universal patterns that an isolated doctor could never see.